# Arabic Hate Speech Classification (AraBERT)

**Scope:** this notebook trains a classifier for **Arabic hate speech in text** (`0 = Non-Hate Speech`, `1 = Hate Speech`).

**Requirements:** Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU), a Hugging Face account with access granted to the gated dataset

## 1. Environment setup and project modules

In [6]:
!pip install -q -U "transformers>=4.41" datasets accelerate arabert gradio scikit-learn pandas matplotlib huggingface_hub

In [7]:
import os
from google.colab import drive
drive.mount("/content/drive")                      # approve the popup now, so training is not interrupted later
os.environ["PROJECT_OUT"] = "/content/drive/MyDrive/arabic_hate"   # model, results and model card are saved here
os.makedirs("src", exist_ok=True); open("src/__init__.py", "w").close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The next cells write the project modules (identical to `src/` in the GitHub repo) so this notebook is self-contained.

In [8]:
%%writefile src/config.py
"""Central configuration shared by every module and the notebook."""
import os
from pathlib import Path

SEED = 42
DATASET_NAME = "manueltonneau/arabic-hate-speech-superset"
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
MAX_LENGTH = 128

ID2LABEL = {0: "Non-Hate Speech", 1: "Hate Speech"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

# Set PROJECT_OUT (e.g. a Google Drive folder) BEFORE importing this module to save outputs there.
_OUT = Path(os.environ.get("PROJECT_OUT", "."))
ARTIFACT_DIR = _OUT / "artifacts/arabic_hate_speech_arabert"
RESULTS_DIR = _OUT / "results"
CHECKPOINT_DIR = Path("checkpoints")   # intermediate Trainer checkpoints (git-ignored)
PROCESSED_DIR = Path("data/processed")  # optional saved splits (git-ignored)

TRAIN_CONFIG = {
    "learning_rate": 2e-5,
    "num_epochs": 2,           # epoch 2 was the best epoch in the first 3-epoch run
    "train_batch_size": 32,   # use 16 or 8 if you hit GPU out-of-memory errors
    "eval_batch_size": 128,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "early_stopping_patience": 2,
    "use_arabert_preprocessing": True,
    "use_tuned_threshold": True,  # threshold is tuned on VALIDATION only, then frozen
}

Writing src/config.py


In [9]:
%%writefile src/data_preparation.py
"""Dataset loading, cleaning, deduplication, stratified splitting, undersampling, tokenization."""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import DATASET_NAME, MODEL_NAME, MAX_LENGTH, SEED, PROCESSED_DIR


def load_raw_dataset() -> pd.DataFrame:
    """Load the gated dataset (requires prior `huggingface_hub.login`). All splits are concatenated."""
    from datasets import load_dataset

    ds = load_dataset(DATASET_NAME)
    frames = []
    for split_name, split in ds.items():
        part = split.to_pandas()
        part["hf_split"] = split_name
        frames.append(part)
    return pd.concat(frames, ignore_index=True)


def clean_and_deduplicate(df: pd.DataFrame):
    """Select text/labels, drop missing, cast labels to int, drop exact duplicate texts.

    Returns (clean_df, stats). Duplicate texts keep the first occurrence; the number of texts
    that appear with conflicting labels is reported because it is a direct annotation-noise signal.
    """
    stats = {"rows_raw": int(len(df))}
    df = df[["text", "labels"]].copy()
    df = df.dropna(subset=["text", "labels"])
    df["text"] = df["text"].astype(str).str.strip()      # whitespace trim only
    df = df[df["text"] != ""]
    df["labels"] = df["labels"].astype(int)
    df = df[df["labels"].isin([0, 1])]
    stats["rows_after_missing_removal"] = int(len(df))

    label_counts_per_text = df.groupby("text")["labels"].nunique()
    stats["texts_with_conflicting_labels"] = int((label_counts_per_text > 1).sum())
    stats["exact_duplicate_rows_removed"] = int(df.duplicated(subset="text").sum())

    df = df.drop_duplicates(subset="text", keep="first").reset_index(drop=True)
    stats["rows_after_dedup"] = int(len(df))
    stats["label_counts_after_dedup"] = {int(k): int(v) for k, v in df["labels"].value_counts().sort_index().items()}
    return df, stats


def stratified_split(df: pd.DataFrame, seed: int = SEED):
    """80/10/10 stratified split of the deduplicated data."""
    train, temp = train_test_split(df, test_size=0.20, stratify=df["labels"], random_state=seed)
    val, test = train_test_split(temp, test_size=0.50, stratify=temp["labels"], random_state=seed)
    return (train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True))


def undersample_train(train: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    """Keep all minority examples, sample the same number of majority examples (no replacement), shuffle.
    Apply to the TRAINING split only."""
    minority_label = train["labels"].value_counts().idxmin()
    minority = train[train["labels"] == minority_label]
    majority = train[train["labels"] != minority_label].sample(n=len(minority), replace=False, random_state=seed)
    return pd.concat([minority, majority]).sample(frac=1.0, random_state=seed).reset_index(drop=True)


def verify_splits(train_full, train_bal, val, test) -> dict:
    """Sanity checks: no text leakage between splits, balanced train, imbalanced eval sets."""
    full, v, t = set(train_full["text"]), set(val["text"]), set(test["text"])
    assert not (full & v) and not (full & t) and not (v & t), "Text leakage between splits!"
    assert set(train_bal["text"]) <= full, "Balanced train must be a subset of the train split"
    counts = train_bal["labels"].value_counts()
    assert counts.nunique() == 1, "Balanced training set is not balanced"
    return {
        "train_balanced": {int(k): int(v) for k, v in counts.items()},
        "validation": {int(k): int(v) for k, v in val["labels"].value_counts().sort_index().items()},
        "test": {int(k): int(v) for k, v in test["labels"].value_counts().sort_index().items()},
        "sizes": {"train_full": len(train_full), "train_balanced": len(train_bal), "validation": len(val), "test": len(test)},
        "validation_hate_rate": float(val["labels"].mean()),
        "test_hate_rate": float(test["labels"].mean()),
    }


def prepare_all(raw_df: pd.DataFrame, seed: int = SEED):
    """Convenience wrapper: clean -> split -> undersample -> verify."""
    clean, clean_stats = clean_and_deduplicate(raw_df)
    train_full, val, test = stratified_split(clean, seed)
    train_bal = undersample_train(train_full, seed)
    split_stats = verify_splits(train_full, train_bal, val, test)
    return train_bal, val, test, {**clean_stats, **split_stats}


def save_splits(train_bal, val, test, out_dir=PROCESSED_DIR):
    out_dir.mkdir(parents=True, exist_ok=True)
    train_bal.to_csv(out_dir / "train_balanced.csv", index=False)
    val.to_csv(out_dir / "validation.csv", index=False)
    test.to_csv(out_dir / "test.csv", index=False)


# ---------------------------------------------------------------------------------------------
# AraBERT preprocessing + tokenization
# ---------------------------------------------------------------------------------------------
def get_arabert_preprocessor():
    """Official AraBERT preprocessor for the matching model (light, non-destructive cleaning)."""
    from arabert.preprocess import ArabertPreprocessor
    return ArabertPreprocessor(model_name=MODEL_NAME)


def preprocess_texts(texts, enabled: bool = True):
    if not enabled:
        return list(texts)
    prep = get_arabert_preprocessor()
    return [prep.preprocess(t) for t in texts]


def make_hf_dataset(df: pd.DataFrame, tokenizer, use_arabert_preprocessing: bool = True):
    """Preprocess + tokenize a dataframe (columns text, labels). Row order is preserved,
    so predictions can be joined back to the original, unprocessed text."""
    from datasets import Dataset

    texts = preprocess_texts(df["text"].tolist(), use_arabert_preprocessing)
    ds = Dataset.from_dict({"text": texts, "labels": df["labels"].astype(int).tolist()})
    ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LENGTH),
                batched=True, remove_columns=["text"])
    return ds


def sort_by_length(ds):
    """Return (dataset sorted by token length, order). Sorted batches need far less padding, which makes
    evaluation/prediction several times faster. `order[k]` is the original row index of sorted row k."""
    lengths = np.fromiter((len(x) for x in ds["input_ids"]), dtype=int)
    order = np.argsort(lengths, kind="stable")
    return ds.select(order.tolist()), order

Writing src/data_preparation.py


In [10]:
%%writefile src/train.py
"""AraBERT fine-tuning with the Hugging Face Trainer (compatible across Transformers versions)."""
import inspect
import json
import math
import platform

import numpy as np
import torch
import transformers
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

from src.data_preparation import sort_by_length
from src.config import (ARTIFACT_DIR, CHECKPOINT_DIR, ID2LABEL, LABEL2ID, MAX_LENGTH, MODEL_NAME,
                        SEED, TRAIN_CONFIG)


def init_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    p, r, f, _ = precision_recall_fscore_support(labels, preds, labels=[1], zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
        "hate_precision": float(p[0]), "hate_recall": float(r[0]), "hate_f1": float(f[0]),
    }


def build_training_args(cfg=None, output_dir=CHECKPOINT_DIR, num_train_examples=None):
    """Build TrainingArguments, adapting argument names to the installed Transformers version."""
    cfg = {**TRAIN_CONFIG, **(cfg or {})}
    params = inspect.signature(TrainingArguments.__init__).parameters
    use_cuda = torch.cuda.is_available()
    bf16 = use_cuda and torch.cuda.is_bf16_supported()
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=cfg["learning_rate"],
        num_train_epochs=cfg["num_epochs"],
        per_device_train_batch_size=cfg["train_batch_size"],
        per_device_eval_batch_size=cfg["eval_batch_size"],
        weight_decay=cfg["weight_decay"],
        save_strategy="epoch",
        save_only_model=True,               # skip optimizer state: faster, smaller checkpoints
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",   # robust to imbalance; validation set keeps natural distribution
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        seed=SEED, data_seed=SEED,
        fp16=use_cuda and not bf16,
        bf16=bf16,
        fp16_full_eval=use_cuda and not bf16,   # faster evaluation
        report_to="none",
    )
    kwargs["eval_strategy" if "eval_strategy" in params else "evaluation_strategy"] = "epoch"

    # warmup: older versions have warmup_ratio, Transformers 5 does not -> use explicit warmup_steps
    if "warmup_ratio" in params:
        kwargs["warmup_ratio"] = cfg["warmup_ratio"]
    elif "warmup_steps" in params and num_train_examples:
        total_steps = math.ceil(num_train_examples / cfg["train_batch_size"]) * cfg["num_epochs"]
        kwargs["warmup_steps"] = int(cfg["warmup_ratio"] * total_steps)

    # batch by similar length (much less padding -> faster); the option name differs across versions
    if "group_by_length" in params:
        kwargs["group_by_length"] = True
    elif "train_sampling_strategy" in params:
        kwargs["train_sampling_strategy"] = "group_by_length"

    def make(kw):
        dropped = [k for k in kw if k not in params]
        if dropped:
            print(f"[compat] TrainingArguments in transformers {transformers.__version__} lacks: {dropped} (skipped)")
        return TrainingArguments(**{k: v for k, v in kw.items() if k in params})

    try:
        return make(kwargs)
    except (TypeError, ValueError) as exc:
        print("[compat] length grouping disabled:", exc)
        for k in ("group_by_length", "train_sampling_strategy"):
            kwargs.pop(k, None)
        return make(kwargs)


def build_trainer(model, tokenizer, train_ds, val_ds, cfg=None):
    cfg = {**TRAIN_CONFIG, **(cfg or {})}
    trainer_params = inspect.signature(Trainer.__init__).parameters
    eval_sorted, _ = sort_by_length(val_ds)   # same metrics, far less padding during per-epoch evaluation
    kwargs = dict(
        model=model, args=build_training_args(cfg, num_train_examples=len(train_ds)),
        train_dataset=train_ds, eval_dataset=eval_sorted,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg["early_stopping_patience"])],
    )
    kwargs["processing_class" if "processing_class" in trainer_params else "tokenizer"] = tokenizer
    return Trainer(**kwargs)


def save_artifacts(trainer, tokenizer, cfg=None, data_stats=None, artifact_dir=ARTIFACT_DIR):
    """Save best model (already loaded by load_best_model_at_end), tokenizer, label map, config."""
    cfg = {**TRAIN_CONFIG, **(cfg or {})}
    artifact_dir.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(artifact_dir))
    tokenizer.save_pretrained(str(artifact_dir))
    (artifact_dir / "label_mapping.json").write_text(
        json.dumps({"id2label": ID2LABEL, "label2id": LABEL2ID}, ensure_ascii=False, indent=2), encoding="utf-8")
    state = trainer.state
    record = {
        "base_model": MODEL_NAME, "seed": SEED, "max_length": MAX_LENGTH, **cfg,
        "best_checkpoint": state.best_model_checkpoint,
        "best_validation_macro_f1": state.best_metric,
        "epochs_completed": state.epoch,
        "versions": {"python": platform.python_version(), "torch": torch.__version__,
                     "transformers": transformers.__version__},
        "data_stats": data_stats,
        "log_history": state.log_history,
    }
    (artifact_dir / "training_config.json").write_text(json.dumps(record, indent=2, default=str), encoding="utf-8")
    return record


def main():
    """End-to-end pipeline: python -m src.train  (needs `huggingface-cli login` first)."""
    from src import data_preparation as dp
    from src import evaluate as ev

    set_seed(SEED)
    raw = dp.load_raw_dataset()
    train_bal, val, test, stats = dp.prepare_all(raw)
    print(json.dumps(stats, indent=2))
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    use_prep = TRAIN_CONFIG["use_arabert_preprocessing"]
    train_ds = dp.make_hf_dataset(train_bal, tokenizer, use_prep)
    val_ds = dp.make_hf_dataset(val, tokenizer, use_prep)
    test_ds = dp.make_hf_dataset(test, tokenizer, use_prep)
    trainer = build_trainer(init_model(), tokenizer, train_ds, val_ds)
    trainer.train()
    save_artifacts(trainer, tokenizer, data_stats=stats)   # save right after training
    ev.evaluate_and_save(trainer, val_ds, test_ds, val, test, stats)


if __name__ == "__main__":
    main()

Writing src/train.py


In [11]:
%%writefile src/evaluate.py
"""Evaluation on validation/test, threshold tuning (validation only), error analysis, result export."""
import json
import re
import shutil

import numpy as np
import pandas as pd
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, f1_score,
                             precision_recall_fscore_support, roc_auc_score)

from src.config import ARTIFACT_DIR, ID2LABEL, RESULTS_DIR, SEED, TRAIN_CONFIG
from src.data_preparation import sort_by_length


def hate_probability(logits: np.ndarray) -> np.ndarray:
    """Softmax probability of class 1 (Hate Speech)."""
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return (e / e.sum(axis=1, keepdims=True))[:, 1]


def predict_proba(trainer, ds, batch_size=256):
    model = trainer.model.eval()
    tok = getattr(trainer, "processing_class", None) or getattr(trainer, "tokenizer", None)
    collator = DataCollatorWithPadding(tok)
    device = next(model.parameters()).device
    sorted_ds, order = sort_by_length(ds)
    keep = [c for c in sorted_ds.column_names if c != "labels"]
    probs = np.empty(len(sorted_ds))
    with torch.no_grad():
        for i in range(0, len(sorted_ds), batch_size):
            chunk = sorted_ds[i:i + batch_size]
            feats = [{k: chunk[k][j] for k in keep} for j in range(len(chunk[keep[0]]))]
            batch = {k: v.to(device) for k, v in collator(feats).items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = model(**batch).logits
            probs[i:i + len(feats)] = ev.hate_probability(logits.float().cpu().numpy())
    out = np.empty_like(probs); out[order] = probs
    return out


def tune_threshold(y_val, p_val, n_candidates=500):
    """Pick the threshold maximizing macro F1 on VALIDATION data only. Returns (best_threshold, table).
    Candidates come from the validation score distribution, so the search is never cut off by a fixed grid."""
    cands = np.unique(np.quantile(p_val, np.linspace(0.30, 0.9995, n_candidates)))
    rows = []
    for t in cands:
        pred = (p_val >= t).astype(int)
        p, r, f, _ = precision_recall_fscore_support(y_val, pred, labels=[1], zero_division=0)
        rows.append({"threshold": float(t), "macro_f1": f1_score(y_val, pred, average="macro"),
                     "hate_precision": float(p[0]), "hate_recall": float(r[0]), "hate_f1": float(f[0])})
    table = pd.DataFrame(rows)
    best = float(table.loc[table["macro_f1"].idxmax(), "threshold"])
    return best, table


def compute_full_metrics(y_true, y_pred) -> dict:
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    per_class = {ID2LABEL[i]: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]), "support": int(s[i])}
                 for i in (0, 1)}
    return {
        "accuracy": float((y_true == y_pred).mean()),
        "per_class": per_class,
        "hate_speech_positive_class": per_class[ID2LABEL[1]],
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).tolist(),
    }


def evaluate_split(y_true, p_hate, threshold: float):
    """Metrics at a fixed threshold + threshold-independent ranking metrics."""
    y_pred = (np.asarray(p_hate) >= threshold).astype(int)
    m = compute_full_metrics(y_true, y_pred)
    m["threshold"] = float(threshold)
    m["roc_auc"] = float(roc_auc_score(y_true, p_hate))
    m["average_precision_hate"] = float(average_precision_score(y_true, p_hate))
    return m, y_pred


def classification_report_text(y_true, y_pred) -> str:
    return classification_report(y_true, y_pred, labels=[0, 1], target_names=[ID2LABEL[0], ID2LABEL[1]],
                                 digits=4, zero_division=0)


def plot_confusion_matrix(cm, path, title="Confusion matrix (test set, natural distribution)"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    cm = np.asarray(cm)
    norm = cm / cm.sum(axis=1, keepdims=True)
    names = [ID2LABEL[0], ID2LABEL[1]]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, data, fmt, sub in ((axes[0], cm, "{:,}", "Counts"), (axes[1], norm, "{:.1%}", "Row-normalized (recall)")):
        im = ax.imshow(data, cmap="Blues")
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(names); ax.set_yticklabels(names)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(sub)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center",
                        color="white" if data[i, j] > data.max() / 2 else "black")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    return fig


def summary_table(metrics: dict) -> pd.DataFrame:
    rows = []
    for name, m in metrics["per_class"].items():
        rows.append({"class": name, **m})
    rows.append({"class": "macro avg (F1)", "f1": metrics["macro_f1"]})
    rows.append({"class": "weighted avg (F1)", "f1": metrics["weighted_f1"]})
    rows.append({"class": "accuracy", "f1": metrics["accuracy"]})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------------------------
# Error analysis: AUTOMATIC hints only. Categories must be assigned by a human reviewer.
# ---------------------------------------------------------------------------------------------
_NEGATION = {"لا", "لم", "لن", "ليس", "ليست", "ما", "مو", "مش", "غير", "بدون"}


def automatic_hints(text: str, confidence: float) -> str:
    hints = []
    tokens = text.split()
    if len(tokens) <= 4:
        hints.append("very short text (little context)")
    if any(tok in _NEGATION for tok in tokens):
        hints.append("contains negation marker (may be spurious)")
    if re.search(r"(.)\1{2,}", text):
        hints.append("character elongation (possible dialect/spelling variation)")
    if re.search(r"[A-Za-z]", text):
        hints.append("contains Latin letters (code-switching / transliteration)")
    if "@" in text or "http" in text:
        hints.append("contains mention/URL")
    if confidence >= 0.95:
        hints.append("high-confidence error (check for annotation inconsistency)")
    return "; ".join(hints)


def build_error_table(test_df: pd.DataFrame, p_hate, threshold: float) -> pd.DataFrame:
    """All misclassified test examples. `test_df` must be in the same row order as the predictions."""
    p_hate = np.asarray(p_hate)
    pred = (p_hate >= threshold).astype(int)
    df = test_df.reset_index(drop=True).copy()
    df["pred"] = pred
    df["p_hate"] = p_hate
    errors = df[df["pred"] != df["labels"]].copy()
    errors["confidence"] = np.where(errors["pred"] == 1, errors["p_hate"], 1 - errors["p_hate"])
    errors["error_type"] = np.where(errors["pred"] == 1, "False positive", "False negative")
    errors["true_label"] = errors["labels"].map(ID2LABEL)
    errors["pred_label"] = errors["pred"].map(ID2LABEL)
    errors["auto_hints"] = [automatic_hints(t, c) for t, c in zip(errors["text"], errors["confidence"])]
    errors["human_review_category"] = ""   # to be filled in manually
    errors["human_verified"] = False
    cols = ["text", "true_label", "pred_label", "p_hate", "confidence", "error_type",
            "auto_hints", "human_review_category", "human_verified"]
    return errors[cols].reset_index(drop=True)


def sample_errors(errors: pd.DataFrame, n_per_type: int = 10, seed: int = SEED) -> pd.DataFrame:
    """Random (not cherry-picked) sample of up to n_per_type false positives and false negatives."""
    parts = []
    for _, grp in errors.groupby("error_type"):
        parts.append(grp.sample(n=min(n_per_type, len(grp)), random_state=seed))
    return pd.concat(parts).reset_index(drop=True) if parts else errors


# ---------------------------------------------------------------------------------------------
# Export
# ---------------------------------------------------------------------------------------------
def write_outputs(metrics: dict, report_text: str, errors_sample: pd.DataFrame, fig_path=None,
                  results_dir=RESULTS_DIR, artifact_dir=ARTIFACT_DIR):
    results_dir.mkdir(parents=True, exist_ok=True)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    (results_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
    (results_dir / "classification_report.txt").write_text(report_text, encoding="utf-8")
    errors_sample.to_csv(results_dir / "error_analysis.csv", index=False, encoding="utf-8-sig")
    (artifact_dir / "decision_rule.json").write_text(json.dumps({
        "positive_label_id": 1, "threshold": metrics["decision_threshold"],
        "rule": "predict Hate Speech if softmax P(class=1) >= threshold",
        "threshold_source": metrics["threshold_source"]}, indent=2), encoding="utf-8")
    eval_dir = artifact_dir / "evaluation"
    eval_dir.mkdir(exist_ok=True)
    for name in ("metrics.json", "classification_report.txt", "error_analysis.csv"):
        shutil.copy(results_dir / name, eval_dir / name)
    if fig_path is not None and fig_path.exists():
        shutil.copy(fig_path, eval_dir / fig_path.name)


def write_model_card(metrics: dict, data_stats: dict, path=None):
    """Model card whose numbers come only from metrics.json produced by the actual run."""
    path = path or (ARTIFACT_DIR / "MODEL_CARD.md")
    t = metrics["test"]
    h = t["hate_speech_positive_class"]
    card = f"""# Arabic Hate Speech Classifier (AraBERT v0.2 base, fine-tuned)

**Scope:** detects *Arabic hate speech in text only* (binary: Hate Speech / Non-Hate Speech).
It does NOT detect bullying, harassment, violence or sexual content, and it is not a video classifier.

- Base model: `aubmindlab/bert-base-arabertv02`
- Data: `manueltonneau/arabic-hate-speech-superset` (gated; verify license/terms before use)
- Training: undersampled-balanced train split; validation/test keep the natural class distribution
- Decision threshold: {metrics['decision_threshold']} ({metrics['threshold_source']})

## Test results (natural distribution, n={sum(data_stats['test'].values()):,})
| Metric | Value |
|---|---|
| Accuracy | {t['accuracy']:.4f} |
| Macro F1 | {t['macro_f1']:.4f} |
| Weighted F1 | {t['weighted_f1']:.4f} |
| Hate Speech precision | {h['precision']:.4f} |
| Hate Speech recall | {h['recall']:.4f} |
| Hate Speech F1 | {h['f1']:.4f} |
| ROC-AUC | {t['roc_auc']:.4f} |
| Average precision (Hate) | {t['average_precision_hate']:.4f} |

## Limitations
Annotation noise, source/political/dialect bias, context-dependent and sarcastic language, and
precision that depends on the very low hate prevalence (~3.7%) of the evaluation data. Predictions
are not definitive judgments and must not be used for automated enforcement without human review.
"""
    path.write_text(card, encoding="utf-8")
    return card


def evaluate_and_save(trainer, val_ds, test_ds, val_df, test_df, data_stats=None, cfg=None):
    """Full evaluation: tune threshold on validation, freeze, evaluate test once, save everything."""
    cfg = {**TRAIN_CONFIG, **(cfg or {})}
    p_val = predict_proba(trainer, val_ds)
    p_test = predict_proba(trainer, test_ds)
    y_val, y_test = val_df["labels"].to_numpy(), test_df["labels"].to_numpy()

    best_t, _ = tune_threshold(y_val, p_val)
    threshold = best_t if cfg["use_tuned_threshold"] else 0.5
    source = "tuned on validation macro F1, then frozen" if cfg["use_tuned_threshold"] else "default 0.5"

    val_m, _ = evaluate_split(y_val, p_val, threshold)
    test_m, y_pred = evaluate_split(y_test, p_test, threshold)
    test_default, _ = evaluate_split(y_test, p_test, 0.5)
    metrics = {"decision_threshold": threshold, "threshold_source": source, "validation": val_m,
               "test": test_m, "test_at_default_threshold_0.5": test_default, "data": data_stats}
    report = classification_report_text(y_test, y_pred)
    errors = build_error_table(test_df, p_test, threshold)
    fig_path = RESULTS_DIR / "confusion_matrix.png"
    RESULTS_DIR.mkdir(exist_ok=True)
    plot_confusion_matrix(test_m["confusion_matrix"], fig_path)
    write_outputs(metrics, report, sample_errors(errors, 15), fig_path)
    write_model_card(metrics, data_stats or {"test": {0: 0, 1: 0}})
    print(report)
    return metrics

Writing src/evaluate.py


In [12]:
%%writefile src/predict.py
"""Load the saved fine-tuned model and run inference (no retraining needed)."""
import json
from pathlib import Path

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import ARTIFACT_DIR, ID2LABEL, MAX_LENGTH
from src.data_preparation import get_arabert_preprocessor


class HateSpeechClassifier:
    def __init__(self, artifact_dir=ARTIFACT_DIR, device=None):
        artifact_dir = Path(artifact_dir)
        if not (artifact_dir / "config.json").exists():
            raise FileNotFoundError(f"No saved model in {artifact_dir}. Train first or restore the artifacts folder.")
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = AutoTokenizer.from_pretrained(artifact_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(artifact_dir).to(self.device).eval()

        cfg_path = artifact_dir / "training_config.json"
        cfg = json.loads(cfg_path.read_text(encoding="utf-8")) if cfg_path.exists() else {}
        self.max_length = cfg.get("max_length", MAX_LENGTH)
        use_prep = cfg.get("use_arabert_preprocessing", True)   # must match training
        self.preprocessor = get_arabert_preprocessor() if use_prep else None

        rule_path = artifact_dir / "decision_rule.json"
        self.threshold = json.loads(rule_path.read_text())["threshold"] if rule_path.exists() else 0.5

    @torch.no_grad()
    def predict(self, texts, batch_size: int = 32):
        single = isinstance(texts, str)
        texts = [texts] if single else list(texts)
        results = []
        for i in range(0, len(texts), batch_size):
            chunk = texts[i:i + batch_size]
            proc = [self.preprocessor.preprocess(t) for t in chunk] if self.preprocessor else chunk
            enc = self.tokenizer(proc, truncation=True, max_length=self.max_length, padding=True,
                                 return_tensors="pt").to(self.device)
            probs = torch.softmax(self.model(**enc).logits, dim=-1).cpu().numpy()
            for text, p in zip(chunk, probs):
                label_id = int(p[1] >= self.threshold)
                results.append({"text": text, "label_id": label_id, "label": ID2LABEL[label_id],
                                "p_hate": float(p[1]),
                                "probabilities": {ID2LABEL[0]: float(p[0]), ID2LABEL[1]: float(p[1])}})
        return results[0] if single else results


if __name__ == "__main__":
    import sys
    clf = HateSpeechClassifier()
    print(json.dumps(clf.predict(" ".join(sys.argv[1:]) or "شكرا لك"), ensure_ascii=False, indent=2))

Writing src/predict.py


In [13]:
%%writefile src/app.py
"""Gradio demo: python -m src.app  (loads the saved model; never retrains)."""
import gradio as gr

from src.config import ARTIFACT_DIR, ID2LABEL
from src.predict import HateSpeechClassifier

DISCLAIMER = (
    "**Disclaimer:** predictions are automatic and are *not* definitive judgments. The model was trained on an "
    "annotated Arabic hate-speech dataset with known noise and may fail on ambiguous, sarcastic, dialectal or "
    "context-dependent text. It detects Arabic hate speech in text only. Do not use it for automated enforcement "
    "without human review."
)
# Illustrative inputs only - NOT verified ground truth.
EXAMPLES = [["شكرا لك على مساعدتك، كان يوما رائعا"],
            ["أنا لا أتفق مع هذا الرأي لكنني أحترمه"],
            ["الطقس جميل اليوم في المدينة"]]


def build_demo(artifact_dir=ARTIFACT_DIR):
    clf = HateSpeechClassifier(artifact_dir)

    def classify(text):
        if not text or not text.strip():
            return "Please enter some Arabic text.", None
        try:
            r = clf.predict(text.strip())
            return f"{r['label']}  (decision threshold on P(Hate) = {clf.threshold:.2f})", r["probabilities"]
        except Exception as exc:  # keep the demo alive on any failure
            return f"Error: {exc}", None

    with gr.Blocks(title="Arabic Hate Speech Classifier") as demo:
        gr.Markdown("# Arabic Hate Speech Classifier (AraBERT)\nPhase 1 of a multimodal abuse-detection project.")
        inp = gr.Textbox(label="Arabic text", lines=4, rtl=True, placeholder="الصق النص العربي هنا...")
        btn = gr.Button("Classify", variant="primary")
        verdict = gr.Textbox(label="Prediction", interactive=False)
        probs = gr.Label(label="Model probabilities", num_top_classes=2)
        gr.Examples(EXAMPLES, inputs=inp, label="Example inputs (illustrative, not verified ground truth)")
        gr.Markdown(DISCLAIMER)
        btn.click(classify, inp, [verdict, probs])
        inp.submit(classify, inp, [verdict, probs])
    return demo


if __name__ == "__main__":
    build_demo().launch(share=False)

Writing src/app.py


In [14]:
import sys
sys.path.insert(0, ".")
import pandas as pd, numpy as np, torch
from src.config import *
CFG = dict(TRAIN_CONFIG)   # edit e.g. CFG["train_batch_size"] = 16 if you run out of GPU memory
print(CFG); print("outputs ->", ARTIFACT_DIR)

{'learning_rate': 2e-05, 'num_epochs': 2, 'train_batch_size': 32, 'eval_batch_size': 128, 'weight_decay': 0.01, 'warmup_ratio': 0.06, 'early_stopping_patience': 2, 'use_arabert_preprocessing': True, 'use_tuned_threshold': True}
outputs -> /content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert


## 2. GPU availability check

In [15]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"- {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU: training on CPU is impractically slow. Switch the runtime to a GPU.")

CUDA available: True
Tesla T4 - 15.6 GB


## 3. Hugging Face authentication

In [16]:
import getpass
from huggingface_hub import login
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass
token = token or os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (input hidden): ")
login(token=token, add_to_git_credential=False)
del token

Hugging Face token (input hidden): ··········


## 4. Dataset download and inspection

In [17]:
from src.data_preparation import load_raw_dataset
raw_df = load_raw_dataset()
print(raw_df.shape); print(raw_df.columns.tolist())
raw_df.head()

README.md:   0%|          | 0.00/7.98k [00:00<?, ?B/s]

ar_hf_112024.csv: reconstructing file:   0%|          |  0.00B / 76.3MB            

ar_hf_112024.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/449078 [00:00<?, ? examples/s]

(449078, 7)
['text', 'labels', 'source', 'dataset', 'nb_annotators', 'post_author_country_location', 'hf_split']


,text,labels,source,dataset,nb_annotators,post_author_country_location,hf_split
0,الوزير جبران باسيل تاج راسك يا جربان ممنوع بعد...,0.0,Twitter,L-HSAB,3,unknown,train
1,صديقي انت ابن جامعه اللعبه اكبر من داعش اللعبه...,0.0,Twitter,L-HSAB,3,unknown,train
2,و مصلحة لبنان تبدأ باستخراج النفط و الغاز لوقف...,0.0,Twitter,L-HSAB,3,unknown,train
3,وليد جنبلاط كاتب الحكمة يا قذر,0.0,Twitter,L-HSAB,3,unknown,train
4,شو بتلبقلك كلمة خنزير بتجي مفصله على قياسك وشك...,0.0,Twitter,L-HSAB,3,unknown,train


## 5. Label distribution and sample inspection


In [18]:
pd.set_option("display.max_colwidth", 200)
print(raw_df["labels"].value_counts().sort_index().to_string(), "| hate rate: %.2f%%" % (100 * (raw_df["labels"] == 1).mean()))
if "dataset" in raw_df.columns:
    display(raw_df.groupby("dataset")["labels"].agg(n="count", hate_rate="mean").sort_values("n", ascending=False))
for lab in (0, 1):   # raw random samples, not curated
    display(raw_df[raw_df["labels"] == lab][["text"]].sample(5, random_state=SEED))

labels
0.0    432532
1.0     16546 | hate rate: 3.68%


,n,hate_rate
dataset,,
jhsc,398662,0.017371
OSACT,12363,0.105395
aracovid,8701,0.087116
T-HSAB,5964,0.179074
L-HSAB,5754,0.080813
Let-Mi,5229,0.493402
MLMA,3352,0.224940
saudi,3178,0.267464
brothers,2946,0.400883


,text
349523,رُغم قُبلته الإعتذار لها ، لكنها ما زالت مُعرضةً عن الصُلح . . ! @ مدينة الحصن
170323,قال رسول الله ﷺ : . عليكَ بالرِّفقِ ؛ فإنَّهُ لا يكونُ في شَيءٍ إلاَّ زانَهُ ، ولا يُنزَعُ مِن شيءٍ إلَّا شانَهُ . رواه مسلم . وقال عليه الصلاة والسلام : . إنَّ اللهَ رفيقٌ يحبُّ الرِّفقَ . ويُعط...
212554,و لا المانيا ولا بريطانيا و لا امريكا ولا روسيا بتيجي مع الاردن بكفة .. ولا اي دوله عربية ولا اجنبيه ولا الدول الشرقيه ..
255543,اي كذبه فيهم
269224,شكرًا للناس اللي معانا بدون أسباب،بدون شروط،بدون مصالح،شكرًا للناس اللي بتهون علينا دايما اللي بتحبنا جد من غير مقابل،اللي طول ما الواحد مقصر في حقها مبتزعل ،شكرًا للناس اللي بتحب تشوفك كويس ومرتا...


,text
412713,@USER @USER اها حماس هم الخونه مو العرب الي خانو حماس الله يلعن بول البعير
412031,بيتعاطوا بول البعير حمقى بني سعود @USER
406387,ﻻ ﻻ ﻻ ﻻ ﻻ ﻻ ﻻ ﻻ احباءنا تونس ينكر نصا صريحا كتاب كافر
445232,لحية العيال وحواجب البنات اذا ريّشت اعرف ان النفسية زفت .! 😏😤🙄💔! #بايع_الكليجا
191972,هههههههههههههههههه نو يوزي انا xp xp سوري عالهبل xp


## 6. Cleaning, deduplication and stratified 80/10/10 split
Rows with missing text/labels are dropped, labels cast to int, exact duplicate texts removed *before* splitting.

In [19]:
from src.data_preparation import clean_and_deduplicate, stratified_split
clean_df, clean_stats = clean_and_deduplicate(raw_df)
print(clean_stats)
train_full, val_df, test_df = stratified_split(clean_df, SEED)
for name, d in [("train_full", train_full), ("validation", val_df), ("test", test_df)]:
    print(f"{name:11s} n={len(d):,}  hate rate={d['labels'].mean():.4%}")

{'rows_raw': 449078, 'rows_after_missing_removal': 449078, 'texts_with_conflicting_labels': 49, 'exact_duplicate_rows_removed': 1454, 'rows_after_dedup': 447624, 'label_counts_after_dedup': {0: 431117, 1: 16507}}
train_full  n=358,099  hate rate=3.6878%
validation  n=44,762  hate rate=3.6862%
test        n=44,763  hate rate=3.6883%


## 7. Training-only undersampling
All minority examples are kept; an equal number of majority examples is sampled without replacement. **Validation and test are untouched.** Trade-off: undersampling discards majority-class information, and because the real data is ~96% non-hate, precision on the natural test distribution can be much lower than recall.

In [20]:
from src.data_preparation import undersample_train, verify_splits
train_bal = undersample_train(train_full, SEED)
split_stats = verify_splits(train_full, train_bal, val_df, test_df)   # asserts no leakage + balanced train
data_stats = {**clean_stats, **split_stats}
print(split_stats)

{'train_balanced': {1: 13206, 0: 13206}, 'validation': {0: 43112, 1: 1650}, 'test': {0: 43112, 1: 1651}, 'sizes': {'train_full': 358099, 'train_balanced': 26412, 'validation': 44762, 'test': 44763}, 'validation_hate_rate': 0.036861623698672984, 'test_hate_rate': 0.036883140093380694}


## 8. Preprocessing and tokenization
Official AraBERT preprocessor for `bert-base-arabertv02` (light cleaning; dialects, negation and punctuation are kept; no stop-word removal).

In [21]:
from transformers import AutoTokenizer
from src.data_preparation import make_hf_dataset
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = make_hf_dataset(train_bal, tokenizer, CFG["use_arabert_preprocessing"])
val_ds   = make_hf_dataset(val_df,   tokenizer, CFG["use_arabert_preprocessing"])
test_ds  = make_hf_dataset(test_df,  tokenizer, CFG["use_arabert_preprocessing"])
lens = np.array([len(x) for x in train_ds["input_ids"]])
print(f"token length: median={np.median(lens):.0f}, p95={np.percentile(lens,95):.0f}, truncated at {MAX_LENGTH}: {(lens >= MAX_LENGTH).mean():.2%}")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/26412 [00:00<?, ? examples/s]

Map:   0%|          | 0/44762 [00:00<?, ? examples/s]

Map:   0%|          | 0/44763 [00:00<?, ? examples/s]

token length: median=20, p95=56, truncated at 128: 0.03%


## 9. AraBERT model initialization

In [22]:
from transformers import set_seed
from src.train import init_model
set_seed(SEED)
model = init_model()
print(model.config.id2label, "| parameters: %.1fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: 'Non-Hate Speech', 1: 'Hate Speech'} | parameters: 135.2M


## 10. Fine-tuning


In [23]:
from src.train import build_trainer, save_artifacts
trainer = build_trainer(model, tokenizer, train_ds, val_ds, CFG)
train_result = trainer.train()
print(train_result.metrics)
save_artifacts(trainer, tokenizer, CFG, data_stats)   # best model + tokenizer + config -> Google Drive, right after training
print("model saved to:", ARTIFACT_DIR)

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,Hate Precision,Hate Recall,Hate F1
1,0.370525,0.413042,0.821210,0.584065,0.874976,0.158918,0.896970,0.269999
2,0.303856,0.371322,0.841651,0.604214,0.888166,0.177920,0.910303,0.297662


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 549.7731, 'train_samples_per_second': 96.083, 'train_steps_per_second': 3.005, 'total_flos': 672454301956800.0, 'train_loss': 0.3813983435780892, 'epoch': 2.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

model saved to: /content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert


In [28]:
import torch
import src.evaluate as ev
from transformers import DataCollatorWithPadding
from sklearn.metrics import roc_auc_score, recall_score
from src.data_preparation import sort_by_length

def predict_proba(trainer, ds, batch_size=256):
    model = trainer.model.eval()
    tok = getattr(trainer, "processing_class", None) or getattr(trainer, "tokenizer", None)
    collator = DataCollatorWithPadding(tok)
    device = next(model.parameters()).device
    sorted_ds, order = sort_by_length(ds)
    keep = [c for c in sorted_ds.column_names if c != "labels"]
    probs = np.empty(len(sorted_ds))
    with torch.no_grad():
        for i in range(0, len(sorted_ds), batch_size):
            chunk = sorted_ds[i:i + batch_size]
            feats = [{k: chunk[k][j] for k in keep} for j in range(len(chunk[keep[0]]))]
            batch = {k: v.to(device) for k, v in collator(feats).items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = model(**batch).logits
            probs[i:i + len(feats)] = ev.hate_probability(logits.float().cpu().numpy())
    out = np.empty_like(probs); out[order] = probs
    return out

ev.predict_proba = predict_proba   # your notebook cells import it from src.evaluate

chk = predict_proba(trainer, val_ds)
print("val ROC-AUC:", round(roc_auc_score(y_val, chk), 4),
      "| hate recall @0.5:", round(recall_score(y_val, (chk >= 0.5).astype(int)), 4), "(training log: 0.9024)")

val ROC-AUC: 0.9407 | hate recall @0.5: 0.9109 (training log: 0.9024)


## 11. Validation and checkpoint selection


In [26]:
from src.evaluate import predict_proba, tune_threshold, evaluate_split
print("best checkpoint:", trainer.state.best_model_checkpoint, "| best val macro F1:", trainer.state.best_metric)
display(pd.DataFrame([l for l in trainer.state.log_history if "eval_macro_f1" in l]))
y_val = val_df["labels"].to_numpy(); y_test = test_df["labels"].to_numpy()
p_val = predict_proba(trainer, val_ds)
RESULTS_DIR.mkdir(parents=True, exist_ok=True); np.save(RESULTS_DIR / "p_val.npy", p_val)
best_t, thr_table = tune_threshold(y_val, p_val)
THRESHOLD = best_t if CFG["use_tuned_threshold"] else 0.5
print("tuned threshold (validation):", best_t, "| threshold used:", THRESHOLD)
print("best threshold at the edge of the search range:", thr_table["macro_f1"].idxmax() == len(thr_table) - 1)
display(thr_table.iloc[::50])

best checkpoint: checkpoints/checkpoint-1652 | best val macro F1: 0.6042139779034394


,eval_loss,eval_accuracy,eval_macro_f1,eval_weighted_f1,eval_hate_precision,eval_hate_recall,eval_hate_f1,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch,step
0,0.413042,0.821210,0.584065,0.874976,0.158918,0.896970,0.269999,86.0252,520.336,4.069,1.0,826
1,0.371322,0.841651,0.604214,0.888166,0.177920,0.910303,0.297662,87.6667,510.593,3.992,2.0,1652


tuned threshold (validation): 0.9482424542172883 | threshold used: 0.9482424542172883
best threshold at the edge of the search range: False


,threshold,macro_f1,hate_precision,hate_recall,hate_f1
0,0.025565,0.286440,0.052273,0.993939,0.099322
50,0.037047,0.351299,0.061003,0.990909,0.114930
100,0.053403,0.399099,0.070297,0.986667,0.131243
150,0.078078,0.441509,0.081672,0.980606,0.150785
200,0.117978,0.481515,0.096224,0.971515,0.175105
250,0.195594,0.523241,0.116694,0.955758,0.207993
300,0.343618,0.570434,0.148516,0.933939,0.256278
350,0.616474,0.626935,0.201353,0.883636,0.327972
400,0.853913,0.695328,0.301217,0.750303,0.429861
450,0.965119,0.725867,0.629744,0.372121,0.467810


In [27]:
from sklearn.metrics import roc_auc_score
print("val ROC-AUC:", round(roc_auc_score(y_val, p_val), 4))

val ROC-AUC: 0.9407


## 12. Final test evaluation

In [29]:
from src.evaluate import classification_report_text, plot_confusion_matrix, summary_table
p_test = predict_proba(trainer, test_ds)
np.save(RESULTS_DIR / "p_test.npy", p_test)
test_m, y_pred = evaluate_split(y_test, p_test, THRESHOLD)
default_m, _ = evaluate_split(y_test, p_test, 0.5)
print(classification_report_text(y_test, y_pred))
display(summary_table(test_m))
h = test_m["hate_speech_positive_class"]
print(f"Hate Speech (positive class): precision={h['precision']:.4f} recall={h['recall']:.4f} f1={h['f1']:.4f}")
print(f"ROC-AUC={test_m['roc_auc']:.4f}  Average precision (Hate)={test_m['average_precision_hate']:.4f}")
print(f"For comparison, at default threshold 0.5: macro F1={default_m['macro_f1']:.4f}, Hate precision={default_m['hate_speech_positive_class']['precision']:.4f}, recall={default_m['hate_speech_positive_class']['recall']:.4f}")
fig = plot_confusion_matrix(test_m["confusion_matrix"], RESULTS_DIR / "confusion_matrix.png")

                 precision    recall  f1-score   support

Non-Hate Speech     0.9794    0.9826    0.9810     43112
    Hate Speech     0.5036    0.4597    0.4807      1651

       accuracy                         0.9634     44763
      macro avg     0.7415    0.7212    0.7308     44763
   weighted avg     0.9618    0.9634    0.9626     44763



,class,precision,recall,f1,support
0,Non-Hate Speech,0.979379,0.982650,0.981011,43112.0
1,Hate Speech,0.503650,0.459721,0.480684,1651.0
2,macro avg (F1),NaN,NaN,0.730848,NaN
3,weighted avg (F1),NaN,NaN,0.962558,NaN
4,accuracy,NaN,NaN,0.963363,NaN


Hate Speech (positive class): precision=0.5036 recall=0.4597 f1=0.4807
ROC-AUC=0.9388  Average precision (Hate)=0.4733
For comparison, at default threshold 0.5: macro F1=0.6012, Hate precision=0.1748, recall=0.8995


## 13. Error analysis
Errors come from real test predictions. The `auto_hints` column is produced by **simple automatic heuristics** (not verified categories). The `human_review_category` column is empty on purpose: read each example and assign a category yourself (e.g. ambiguous political language, insult without targeting, sarcasm, dialect/spelling variation, negation/indirect meaning, annotation inconsistency, non-hateful discussion, implicit hate). The sample is random, not cherry-picked.

In [30]:
from src.evaluate import build_error_table, sample_errors
errors = build_error_table(test_df, p_test, THRESHOLD)
print(f"total errors: {len(errors):,}  ({errors['error_type'].value_counts().to_dict()})")
error_sample = sample_errors(errors, n_per_type=15)
pd.set_option("display.max_colwidth", 160)
display(error_sample[["text", "true_label", "pred_label", "confidence", "error_type", "auto_hints"]])

total errors: 1,640  ({'False negative': 892, 'False positive': 748})


,text,true_label,pred_label,confidence,error_type,auto_hints
0,عاق جسدياً :!يريد أن يشرب الماء فيوقِع الكأس ولا يستطيع أن يرفعه !!! :),Hate Speech,Non-Hate Speech,0.243642,False negative,character elongation (possible dialect/spelling variation)
1,لا كل الثواار كلاب متل النظام هيك جملة سهل قولها وما بتوجع الراس وبتريحك من كتير وجع راس وتفكير الكل كلاب ليش اهتم او ادعم طرف؟,Hate Speech,Non-Hate Speech,0.716814,False negative,contains negation marker (may be spurious)
2,@USER @USER @USER @USER سيدفعون الثمن غالي \nعهدا علينا كا جنوبيين ندفعهم بما قاموا به من هدا العمل قليلي الاصل وناكرين المعروف,Hate Speech,Non-Hate Speech,0.153042,False negative,contains Latin letters (code-switching / transliteration); contains mention/URL
3,اختلفنا مين يقود عالثاني اكثر,Hate Speech,Non-Hate Speech,0.970240,False negative,high-confidence error (check for annotation inconsistency)
4,يا تجار لبنان ..\nوخاصه المواد الأساسيه للمواطن ..\nأنتم أوسخ وأحقر من فيروس كورونا ..\nعيب إستحوا الله لا يشبعكن ..,Hate Speech,Non-Hate Speech,0.210766,False negative,contains negation marker (may be spurious)
5,يا بعدهم لي الشرف,Hate Speech,Non-Hate Speech,0.136608,False negative,very short text (little context)
6,هدير الf16 له عشاقه...فرسان سلاح الجو الملكي يحلقون فوق عمان.,Hate Speech,Non-Hate Speech,0.617224,False negative,character elongation (possible dialect/spelling variation); contains Latin letters (code-switching / transliteration)
7,ونحن وهم نعيش بخير هالوطن بناكل ونشرب نفس بعض وعلى ارض وطني نعيش بأمن وامان اللهم احمي ملكنا ووطني وشعبه واخواننا السوريين,Hate Speech,Non-Hate Speech,0.054601,False negative,
8,السﻻم يطلق على كل شي ،عليك وعلى أهلك وعلى دارك وأحيائك وكل شي السﻻم رحمه وهدؤء وسكينه ومحبه وليس مختصر ع أحد .,Hate Speech,Non-Hate Speech,0.881615,False negative,
9,ضفدع قاعد يشرب قهوة,Hate Speech,Non-Hate Speech,0.126353,False negative,very short text (little context)


## 14. Save model, tokenizer, configuration and evaluation outputs

In [31]:
# the model and tokenizer were already saved to Drive in section 10; this saves the evaluation outputs next to them
from src.evaluate import write_outputs, write_model_card
metrics = {"decision_threshold": THRESHOLD,
           "threshold_source": "tuned on validation macro F1, then frozen" if CFG["use_tuned_threshold"] else "default 0.5",
           "validation": evaluate_split(y_val, p_val, THRESHOLD)[0], "test": test_m,
           "test_at_default_threshold_0.5": default_m, "data": data_stats}
write_outputs(metrics, classification_report_text(y_test, y_pred), error_sample, RESULTS_DIR / "confusion_matrix.png")
write_model_card(metrics, data_stats)
print(sorted(str(p) for p in ARTIFACT_DIR.rglob("*") if p.is_file()))

['/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/MODEL_CARD.md', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/config.json', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/decision_rule.json', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/evaluation/classification_report.txt', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/evaluation/confusion_matrix.png', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/evaluation/error_analysis.csv', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/evaluation/metrics.json', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/label_mapping.json', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/model.safetensors', '/content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert/tokenizer.json', '/content/drive/MyDrive/arabi

In [32]:
import os, sys
sys.path.insert(0, ".")
os.environ["PROJECT_OUT"] = "/content/drive/MyDrive/arabic_hate"   # must be set before importing src.config
from src.config import ARTIFACT_DIR
print("model dir:", ARTIFACT_DIR)
print("model found:", (ARTIFACT_DIR / "config.json").exists())

model dir: /content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert
model found: True


## 15. Reload the saved model for inference
This is also how you would run inference in a fresh runtime: unzip the artifacts, `pip install` the requirements, copy `src/`, and run the cell below (no retraining, no dataset access needed).

In [33]:
from src.predict import HateSpeechClassifier
clf = HateSpeechClassifier(ARTIFACT_DIR)
samples = ["شكرا لك على مساعدتك", "الطقس جميل اليوم"]   # illustrative inputs, not ground truth
for r in clf.predict(samples):
    print(r["label"], {k: round(v, 4) for k, v in r["probabilities"].items()}, "|", r["text"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Non-Hate Speech {'Non-Hate Speech': 0.9569, 'Hate Speech': 0.0431} | شكرا لك على مساعدتك
Non-Hate Speech {'Non-Hate Speech': 0.845, 'Hate Speech': 0.155} | الطقس جميل اليوم


## 16. Interactive Gradio demo
Loads the saved model (no retraining). `share=True` creates a temporary public link; stop the cell to shut it down.

In [34]:
from src.app import build_demo
demo = build_demo(ARTIFACT_DIR)
demo.launch(share=True, debug=False)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ef6f93bca1ac4b3691.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 17. Export results, model card and artifacts

In [35]:
# Everything is already on Google Drive (MyDrive/arabic_hate): the model folder and the results folder.
# To download from Drive, open drive.google.com, right-click the folder -> Download.
from src.config import ARTIFACT_DIR, ID2LABEL, RESULTS_DIR, SEED, TRAIN_CONFIG
print("model:", ARTIFACT_DIR); print("results:", RESULTS_DIR)
print(open(ARTIFACT_DIR / "MODEL_CARD.md", encoding="utf-8").read())

model: /content/drive/MyDrive/arabic_hate/artifacts/arabic_hate_speech_arabert
results: /content/drive/MyDrive/arabic_hate/results
# Arabic Hate Speech Classifier (AraBERT v0.2 base, fine-tuned)

**Scope:** detects *Arabic hate speech in text only* (binary: Hate Speech / Non-Hate Speech).
It does NOT detect bullying, harassment, violence or sexual content, and it is not a video classifier.

- Base model: `aubmindlab/bert-base-arabertv02`
- Data: `manueltonneau/arabic-hate-speech-superset` (gated; verify license/terms before use)
- Training: undersampled-balanced train split; validation/test keep the natural class distribution
- Decision threshold: 0.9482424542172883 (tuned on validation macro F1, then frozen)

## Test results (natural distribution, n=44,763)
| Metric | Value |
|---|---|
| Accuracy | 0.9634 |
| Macro F1 | 0.7308 |
| Weighted F1 | 0.9626 |
| Hate Speech precision | 0.5036 |
| Hate Speech recall | 0.4597 |
| Hate Speech F1 | 0.4807 |
| ROC-AUC | 0.9388 |
| Average precisi

# Training a baseline model
###(TF-IDF + Logistic Regression)

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import hstack

# vectorizers are fitted on the balanced TRAIN split only (no leakage from val/test)
word_vec = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_features=200_000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=300_000, sublinear_tf=True)

def feats(texts, fit=False):
    if fit:
        return hstack([word_vec.fit_transform(texts), char_vec.fit_transform(texts)]).tocsr()
    return hstack([word_vec.transform(texts), char_vec.transform(texts)]).tocsr()

X_tr = feats(train_bal["text"], fit=True)
X_va = feats(val_df["text"]);  X_te = feats(test_df["text"])

lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr.fit(X_tr, train_bal["labels"])
pb_val  = lr.predict_proba(X_va)[:, 1]
pb_test = lr.predict_proba(X_te)[:, 1]

# same procedure as AraBERT: tune threshold on VALIDATION only, then freeze
tb, _ = tune_threshold(y_val, pb_val)
print("baseline tuned threshold:", tb)

baseline tuned threshold: 0.8039238711579736


# Comparing

In [37]:
rows = []
def add(name, p_test_, thr):
    m, _ = evaluate_split(y_test, p_test_, thr)
    h = m["hate_speech_positive_class"]
    rows.append({"model": name, "threshold": round(thr, 4), "accuracy": m["accuracy"], "macro_f1": m["macro_f1"],
                 "hate_precision": h["precision"], "hate_recall": h["recall"], "hate_f1": h["f1"],
                 "roc_auc": m["roc_auc"], "avg_precision_hate": m["average_precision_hate"]})

add("TF-IDF + LR (thr 0.5)",       pb_test, 0.5)
add("TF-IDF + LR (val-tuned thr)", pb_test, tb)
add("AraBERT (thr 0.5)",           p_test,  0.5)
add("AraBERT (val-tuned thr)",     p_test,  THRESHOLD)

cmp_df = pd.DataFrame(rows).set_index("model").round(4)
display(cmp_df)
cmp_df.to_csv(RESULTS_DIR / "baseline_comparison.csv")

,threshold,accuracy,macro_f1,hate_precision,hate_recall,hate_f1,roc_auc,avg_precision_hate
model,,,,,,,,
TF-IDF + LR (thr 0.5),0.5000,0.8708,0.6284,0.2031,0.8565,0.3284,0.9288,0.3601
TF-IDF + LR (val-tuned thr),0.8039,0.9481,0.6938,0.3552,0.4979,0.4146,0.9288,0.3601
AraBERT (thr 0.5),0.5000,0.8397,0.6012,0.1748,0.8995,0.2928,0.9388,0.4733
AraBERT (val-tuned thr),0.9482,0.9634,0.7308,0.5036,0.4597,0.4807,0.9388,0.4733
